# Demo: Reassemble Pipeline

Curation pipeline that searches for events in the org, gathers the matches into a Roboto Collection, and invokes `roboto-to-lerobot-v2_1` against that collection. The contract used for conversion is `contract_demo_reassemble.yaml`, stored in a dedicated Contracts dataset.

1. RoboQL search (default: `tags CONTAINS "to_lerobot"`).
2. Create an event collection (`resource_type="event"`) containing every match.
3. Invoke `roboto-to-lerobot-v2_1` with `collection_id=<id>`.

## Initialization

In [ ]:
import roboto
from notebook_helpers import tail_until_done

# Authentication (API token) is picked up from ~/.roboto/config.json
CONTRACTS_DATASET_ID = "ds_xxxxxxxxxxxx"  # your dataset that holds the contract YAML
CONTRACT = "contract_demo_reassemble.yaml"

roboto_client = roboto.RobotoClient.from_env()
roboto_search = roboto.RobotoSearch.for_roboto_client(roboto_client)

## Search Events and Build a Collection

Customize `EVENT_QUERY` below to whatever RoboQL expression selects the events you want included as episodes. The default — `tags CONTAINS "to_lerobot"` — picks up every event flagged for conversion in the org. You can refine with start-time floors, dataset IDs, metadata fields, etc. Examples:

- `tags CONTAINS "to_lerobot" AND start_time GREATER_THAN 1768343000000000000` — restrict to a recent batch.
- `tags CONTAINS "to_lerobot" AND dataset_id EQUALS "ds_xxxxxxxxxxxx"` — restrict to one source dataset.

Matched event IDs are gathered into a new Roboto Collection whose `resource_type` is `event`. The conversion action then receives this collection by ID, so re-running with the same query produces a fresh collection (and a reproducible record of exactly which events fed each run — captured in the output manifest as `collection_id` + `collection_version`).

In [ ]:
import datetime

# Customize this RoboQL query to select the events you want to convert.
EVENT_QUERY = 'tags CONTAINS "to_lerobot"'

matched_events = list(roboto_search.find_events(EVENT_QUERY))
print(f"Found {len(matched_events)} event(s) matching the query")
assert matched_events, f"No events matched query: {EVENT_QUERY}"

collection = roboto.Collection.create(
    name=f"roboto-to-lerobot demo {datetime.datetime.now(datetime.UTC).isoformat(timespec='seconds')}",
    description=f"Events matched by RoboQL: {EVENT_QUERY!r}",
    resource_type=roboto.CollectionResourceType.Event,
    event_ids=[e.event_id for e in matched_events],
)
COLLECTION_ID = collection.collection_id
print(f"Created collection {COLLECTION_ID} (version {collection.record.version}) with {len(matched_events)} event(s)")

## Create Output Dataset For LeRobot Training Data

In [ ]:
output_dataset = roboto.Dataset.create(
    name="Demo reassemble training dataset",
    description=(
        f"LeRobot dataset for events in collection {COLLECTION_ID} "
        f"(query={EVENT_QUERY!r})"
    ),
    tags=["le_robot_v2_1", "demo"],
    metadata={
        "source_collection_id": COLLECTION_ID,
        "source_collection_version": collection.record.version,
    },
)
print(f"LeRobot dataset to be saved in Roboto dataset '{output_dataset.dataset_id}'")

## Invoke `roboto-to-lerobot-v2_1`

We pass the collection ID we just created; the action loads every event in it and converts each into one episode. The contract lives in its own Contracts dataset (`CONTRACTS_DATASET_ID`); we set that as `data_source_id` so the action can fetch the YAML by relative path. The output is written under `<invocation_id>/combined/` inside `output_dataset`, alongside a self-describing `manifest.json` that records `collection_id` and `collection_version` for reproducibility.

In [ ]:
from roboto.domain import actions

roboto_to_lerobot = actions.Action.from_name("roboto-to-lerobot-v2_1")
convert_iv = roboto_to_lerobot.invoke(
    invocation_source=actions.InvocationSource.Manual,
    data_source_id=CONTRACTS_DATASET_ID,
    input_data=[CONTRACT],
    upload_destination=actions.InvocationUploadDestination.dataset(
        output_dataset.dataset_id
    ),
    parameter_values={
        "collection_id": COLLECTION_ID,
        "contract": CONTRACT,
    },
)

status = tail_until_done(convert_iv)
assert status == actions.InvocationStatus.Completed, (
    f"Conversion invocation did not complete: {status}"
)
print(f"Conversion invocation: {convert_iv.id}")